# Qwen2.5-3B + LLaMA3.2-3B Sentiment Pipeline (Gold Commodity Eval)

This notebook runs three stages:
1. **Zero-shot baseline** with `Qwen/Qwen2.5-3B-Instruct` on the Gold Commodity dataset.
2. **qLoRA fine-tuning** of `unsloth/Llama-3.2-3B-Instruct` on full FinancialPhraseBank, then evaluation on Gold Commodity.
3. **Optuna hyperparameter tuning** for LLaMA qLoRA, then final re-evaluation on Gold Commodity.

All stage metrics are logged to **Weights & Biases** as separate runs for comparison.

In [ ]:
%%capture
!pip install -U unsloth transformers==4.56.2 trl==0.22.2 peft bitsandbytes accelerate datasets scikit-learn matplotlib pandas python-dotenv wandb optuna

## Device Check


In [ ]:
import subprocess
import torch

print("=== nvidia-smi (if available) ===")
try:
    res = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False)
    if res.stdout.strip():
        print(res.stdout)
    else:
        print(res.stderr.strip() or "nvidia-smi returned no output")
except FileNotFoundError:
    print("nvidia-smi not found on this machine.")

if torch.cuda.is_available():
    device_type = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device_type = "mps"
else:
    device_type = "cpu"

print(f"Detected runtime device: {device_type}")
if device_type == "cuda":
    print(f"CUDA device count: {torch.cuda.device_count()}")
    print(f"Current CUDA device: {torch.cuda.current_device()}")
    print(f"CUDA device name: {torch.cuda.get_device_name(torch.cuda.current_device())}")

## 1. Setup and Reproducibility

In [ ]:
import os
import re
import gc
import math
import time
import random
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

def load_dotenv_walk(start: Path) -> bool:
    for parent in [start, *start.parents]:
        env_file = parent / '.env'
        if env_file.exists():
            load_dotenv(env_file, override=False)
            print(f'Loaded .env from: {parent}')
            return True
    return False

_ = load_dotenv_walk(Path(os.getcwd()))

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

print(f'Device: {DEVICE}')
print(f'DType : {DTYPE}')

## 2. W&B + Dataset Loading (Gold + FinancialPhraseBank)

In [ ]:
import wandb
from datasets import load_dataset

WANDB_PROJECT = os.getenv('WANDB_PROJECT', 'is469-sentiment-comparison')
WANDB_ENTITY = os.getenv('WANDB_ENTITY', None)
WANDB_GROUP = os.getenv('WANDB_GROUP', 'gold-commodity-side-by-side')
print(f"W&B destination -> entity={WANDB_ENTITY}, project={WANDB_PROJECT}, group={WANDB_GROUP}")

if not WANDB_PROJECT or not WANDB_ENTITY or not WANDB_GROUP:
    raise ValueError('Set WANDB_PROJECT, WANDB_ENTITY, WANDB_GROUP in a shared .env file at repo root.')

gold_ds = load_dataset('SaguaroCapital/sentiment-analysis-in-commodity-market-gold')
split_name = 'test' if 'test' in gold_ds else 'train'
gold_raw_df = gold_ds[split_name].to_pandas()

sentence_col = next((c for c in ['sentence', 'text', 'content', 'headline'] if c in gold_raw_df.columns), None)
label_col = next((c for c in ['label_text', 'label', 'sentiment', 'target'] if c in gold_raw_df.columns), None)
if sentence_col is None or label_col is None:
    raise ValueError(f'Could not detect sentence/label columns in dataset columns={list(gold_raw_df.columns)}')

labels = gold_raw_df[label_col]
feature = gold_ds[split_name].features.get(label_col)
if hasattr(feature, 'names'):
    label_map = {i: name.lower() for i, name in enumerate(feature.names)}
    label_text = labels.map(lambda x: label_map.get(int(x), str(x).lower()))
else:
    label_text = labels.astype(str).str.strip().str.lower()
    heuristic = {'0': 'negative', '1': 'neutral', '2': 'positive'}
    label_text = label_text.map(lambda x: heuristic.get(x, x))

gold_eval_df = pd.DataFrame({
    'sentence': gold_raw_df[sentence_col].astype(str),
    'label_text': label_text,
})

print(f'Gold eval samples: {len(gold_eval_df)}')
print(gold_eval_df['label_text'].value_counts())

In [ ]:
# Download FinancialPhraseBank (75% agreement)
!curl -L "https://raw.githubusercontent.com/maxwellsarpong/NLP-financial-text-processing-dataset/master/Sentences_75Agree.txt" -o Sentences_75Agree.txt

fpb_df = pd.read_csv(
    'Sentences_75Agree.txt',
    sep='@',
    header=None,
    names=['sentence', 'label_text'],
    engine='python',
    encoding='latin-1',
)
fpb_df['sentence'] = fpb_df['sentence'].str.strip()
fpb_df['label_text'] = fpb_df['label_text'].str.strip().str.lower()

print(f'FPB full samples: {len(fpb_df)}')
print(fpb_df['label_text'].value_counts())

## 3. Shared Helpers (Prompting, Batched Inference, Metrics, Logging)

In [ ]:
from sklearn.metrics import (
    classification_report,
    f1_score,
    accuracy_score,
    matthews_corrcoef,
    confusion_matrix,
)

LABELS = ['negative', 'neutral', 'positive']
LABEL_SET = set(LABELS)

def make_prompt(sentence: str) -> str:
    return (
        'You are a financial sentiment classifier.\n'
        'Return exactly one label: negative, neutral, or positive.\n\n'
        f'Sentence: {sentence}\n'
        'Label:'
    )

def parse_label(text: str) -> str:
    t = text.strip().lower()
    m = re.search(r'(negative|neutral|positive)', t)
    return m.group(1) if m else 'neutral'

@torch.inference_mode()
def batched_generate_labels(model, tokenizer, sentences, batch_size=32, max_new_tokens=3):
    preds = []
    tokenizer.padding_side = 'left'
    prompts = [make_prompt(s) for s in sentences]

    for i in range(0, len(prompts), batch_size):
        chunk = prompts[i : i + batch_size]
        enc = tokenizer(
            chunk,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=512,
        ).to(model.device)

        out = model.generate(
            **enc,
            do_sample=False,
            temperature=None,
            top_p=None,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
        )

        gen_tokens = out[:, enc['input_ids'].shape[1]:]
        texts = tokenizer.batch_decode(gen_tokens, skip_special_tokens=True)
        preds.extend([parse_label(t) for t in texts])

    return preds

def evaluate_predictions(true_labels, pred_labels):
    true_labels = [x.strip().lower() for x in true_labels]
    pred_labels = [x.strip().lower() if x.strip().lower() in LABEL_SET else 'neutral' for x in pred_labels]

    metrics = {
        'macro_f1': f1_score(true_labels, pred_labels, labels=LABELS, average='macro', zero_division=0),
        'micro_f1': f1_score(true_labels, pred_labels, labels=LABELS, average='micro', zero_division=0),
        'weighted_f1': f1_score(true_labels, pred_labels, labels=LABELS, average='weighted', zero_division=0),
        'f1_negative': f1_score(true_labels, pred_labels, labels=['negative'], average='macro', zero_division=0),
        'f1_neutral': f1_score(true_labels, pred_labels, labels=['neutral'], average='macro', zero_division=0),
        'f1_positive': f1_score(true_labels, pred_labels, labels=['positive'], average='macro', zero_division=0),
        'accuracy': accuracy_score(true_labels, pred_labels),
        'mcc': matthews_corrcoef(true_labels, pred_labels),
    }

    metrics['classification_report'] = classification_report(
        true_labels, pred_labels, labels=LABELS, zero_division=0
    )
    metrics['confusion_matrix'] = confusion_matrix(true_labels, pred_labels, labels=LABELS)
    return metrics

def print_metrics(title, metrics):
    print('=' * 80)
    print(title)
    print('=' * 80)
    print(metrics['classification_report'])
    print(f"Macro F1   : {metrics['macro_f1']:.4f}")
    print(f"Micro F1   : {metrics['micro_f1']:.4f}")
    print(f"Weighted F1: {metrics['weighted_f1']:.4f}")
    print(f"Accuracy   : {metrics['accuracy']:.4f}")
    print(f"MCC        : {metrics['mcc']:.4f}")
    print(f"F1 neg     : {metrics['f1_negative']:.4f}")
    print(f"F1 neu     : {metrics['f1_neutral']:.4f}")
    print(f"F1 pos     : {metrics['f1_positive']:.4f}")

def log_run_to_wandb(run_name, config_dict, metrics):
    run = wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        group=WANDB_GROUP,
        name=run_name,
        config=config_dict,
        reinit=True,
    )
    log_metrics = {k: v for k, v in metrics.items() if isinstance(v, (int, float, np.floating))}
    run.log(log_metrics)
    run.summary['classification_report'] = metrics['classification_report']
    run.summary['labels'] = LABELS
    run.finish()

## 4. Baseline: Qwen2.5-3B-Instruct (Non-Finetuned) on Gold Commodity

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

QWEN_MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'

qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME, use_fast=True)
if qwen_tokenizer.pad_token is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_NAME,
    torch_dtype=DTYPE,
    device_map='auto',
    load_in_4bit=True,
)
qwen_model.eval()

qwen_preds = batched_generate_labels(
    qwen_model,
    qwen_tokenizer,
    gold_eval_df['sentence'].tolist(),
    batch_size=64,
    max_new_tokens=3,
)

qwen_metrics = evaluate_predictions(gold_eval_df['label_text'].tolist(), qwen_preds)
print_metrics('Qwen2.5-3B-Instruct (Zero-shot) on Gold Commodity', qwen_metrics)

log_run_to_wandb(
    run_name='qwen2.5-3b-instruct_zero_shot_gold',
    config_dict={
        'model_name': QWEN_MODEL_NAME,
        'stage': 'baseline_zero_shot',
        'dataset_eval': 'gold-commodity-sentiment-eval',
        'quantization': '4bit',
        'batch_size': 64,
    },
    metrics=qwen_metrics,
)

## 5. Fine-tune LLaMA3.2-3B-Instruct (Full FPB), Then Evaluate on Gold

In [ ]:
from datasets import Dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForSeq2Seq

LLAMA_MODEL_NAME = 'unsloth/Llama-3.2-3B-Instruct'
LLAMA_ADAPTER_DIR = 'qlora_adapters/llama3.2-3b-fpb-full-default'
MAX_SEQ_LEN = 1024

def make_chat_train_text(tokenizer, sentence, label):
    messages = [
        {
            'role': 'system',
            'content': 'You are a financial sentiment classifier. Reply with one word: negative, neutral, or positive.',
        },
        {'role': 'user', 'content': f'Classify sentiment:\n{sentence}'},
        {'role': 'assistant', 'content': label},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

llama_model, llama_tokenizer = FastLanguageModel.from_pretrained(
    model_name=LLAMA_MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

llama_model = FastLanguageModel.get_peft_model(
    llama_model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)

train_ds = Dataset.from_dict({
    'text': [make_chat_train_text(llama_tokenizer, s, y) for s, y in zip(fpb_df['sentence'], fpb_df['label_text'])]
})

trainer = SFTTrainer(
    model=llama_model,
    tokenizer=llama_tokenizer,
    train_dataset=train_ds,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    data_collator=DataCollatorForSeq2Seq(tokenizer=llama_tokenizer),
    packing=False,
    args=SFTConfig(
        output_dir='outputs/llama3.2-3b-fpb-full-default',
        num_train_epochs=3,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        weight_decay=0.01,
        optim='adamw_8bit',
        logging_steps=20,
        save_strategy='epoch',
        seed=SEED,
        report_to='none',
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part='<|start_header_id|>user<|end_header_id|>\n\n',
    response_part='<|start_header_id|>assistant<|end_header_id|>\n\n',
)

train_out = trainer.train()
llama_model.save_pretrained(LLAMA_ADAPTER_DIR)
llama_tokenizer.save_pretrained(LLAMA_ADAPTER_DIR)

FastLanguageModel.for_inference(llama_model)
llama_model.eval()

llama_preds = batched_generate_labels(
    llama_model,
    llama_tokenizer,
    gold_eval_df['sentence'].tolist(),
    batch_size=64,
    max_new_tokens=3,
)

llama_metrics = evaluate_predictions(gold_eval_df['label_text'].tolist(), llama_preds)
print_metrics('LLaMA3.2-3B qLoRA (Default Hyperparams) on Gold Commodity', llama_metrics)

log_run_to_wandb(
    run_name='llama3.2-3b-qlora_default_fullfpb_gold',
    config_dict={
        'model_name': LLAMA_MODEL_NAME,
        'adapter_dir': LLAMA_ADAPTER_DIR,
        'stage': 'finetune_default',
        'dataset_train': 'financialphrasebank_75agree_full',
        'dataset_eval': 'gold-commodity-sentiment-eval',
        'lora_r': 16,
        'lora_alpha': 32,
        'lora_dropout': 0.05,
        'epochs': 3,
        'lr': 2e-4,
        'batch_size': 8,
        'grad_accum': 4,
        'train_loss': float(train_out.training_loss),
    },
    metrics=llama_metrics,
)

## 6. Optuna Tuning (LLaMA qLoRA) + Final Evaluation on Gold

To keep tuning fast on GPU clusters, this objective uses fewer epochs and reuses the same evaluation function.

In [ ]:
import optuna

OPTUNA_TRIALS = int(os.getenv('OPTUNA_TRIALS', 8))
OPTUNA_TIMEOUT_SEC = int(os.getenv('OPTUNA_TIMEOUT_SEC', 0))

def objective(trial):
    lora_r = trial.suggest_categorical('lora_r', [8, 16, 32])
    lora_alpha = trial.suggest_categorical('lora_alpha', [16, 32, 64])
    lora_dropout = trial.suggest_float('lora_dropout', 0.0, 0.15)
    lr = trial.suggest_float('learning_rate', 1e-4, 5e-4, log=True)
    batch_size = trial.suggest_categorical('batch_size', [4, 8])
    grad_accum = trial.suggest_categorical('grad_accum', [2, 4])
    epochs = trial.suggest_categorical('epochs', [1, 2])

    model, tok = FastLanguageModel.from_pretrained(
        model_name=LLAMA_MODEL_NAME,
        max_seq_length=MAX_SEQ_LEN,
        dtype=None,
        load_in_4bit=True,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
        bias='none',
        use_gradient_checkpointing='unsloth',
        random_state=SEED,
    )

    trial_out_dir = f'outputs/optuna/trial_{trial.number}'
    trial_trainer = SFTTrainer(
        model=model,
        tokenizer=tok,
        train_dataset=train_ds,
        dataset_text_field='text',
        max_seq_length=MAX_SEQ_LEN,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tok),
        packing=False,
        args=SFTConfig(
            output_dir=trial_out_dir,
            num_train_epochs=epochs,
            per_device_train_batch_size=batch_size,
            gradient_accumulation_steps=grad_accum,
            learning_rate=lr,
            lr_scheduler_type='cosine',
            warmup_ratio=0.03,
            weight_decay=0.01,
            optim='adamw_8bit',
            logging_steps=50,
            save_strategy='no',
            seed=SEED,
            report_to='none',
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
        ),
    )

    trial_trainer = train_on_responses_only(
        trial_trainer,
        instruction_part='<|start_header_id|>user<|end_header_id|>\n\n',
        response_part='<|start_header_id|>assistant<|end_header_id|>\n\n',
    )

    trial_train = trial_trainer.train()
    FastLanguageModel.for_inference(model)
    model.eval()

    trial_preds = batched_generate_labels(
        model,
        tok,
        gold_eval_df['sentence'].tolist(),
        batch_size=64,
        max_new_tokens=3,
    )
    trial_metrics = evaluate_predictions(gold_eval_df['label_text'].tolist(), trial_preds)

    wandb_run = wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        group=WANDB_GROUP,
        name=f'optuna_trial_{trial.number}',
        config={
            'stage': 'optuna_trial',
            'trial_number': trial.number,
            'lora_r': lora_r,
            'lora_alpha': lora_alpha,
            'lora_dropout': lora_dropout,
            'learning_rate': lr,
            'batch_size': batch_size,
            'grad_accum': grad_accum,
            'epochs': epochs,
            'train_loss': float(trial_train.training_loss),
        },
        reinit=True,
    )
    wandb_run.log({
        'macro_f1': trial_metrics['macro_f1'],
        'micro_f1': trial_metrics['micro_f1'],
        'weighted_f1': trial_metrics['weighted_f1'],
        'accuracy': trial_metrics['accuracy'],
        'mcc': trial_metrics['mcc'],
    })
    wandb_run.finish()

    score = trial_metrics['macro_f1']
    trial.set_user_attr('metrics', {k: float(v) for k, v in trial_metrics.items() if isinstance(v, (int, float, np.floating))})

    del model, tok, trial_trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return score

study = optuna.create_study(direction='maximize', study_name='llama3.2-3b-qlora-gold-macrof1')
study.optimize(objective, n_trials=OPTUNA_TRIALS, timeout=(None if OPTUNA_TIMEOUT_SEC == 0 else OPTUNA_TIMEOUT_SEC))

print('Best trial:', study.best_trial.number)
print('Best macro F1:', study.best_value)
print('Best params:', study.best_params)

In [ ]:
# Retrain one final model on full FPB with best Optuna params, then evaluate/log
best = study.best_params
BEST_ADAPTER_DIR = 'qlora_adapters/llama3.2-3b-fpb-optuna-best'

best_model, best_tok = FastLanguageModel.from_pretrained(
    model_name=LLAMA_MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

best_model = FastLanguageModel.get_peft_model(
    best_model,
    r=best['lora_r'],
    lora_alpha=best['lora_alpha'],
    lora_dropout=best['lora_dropout'],
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)

best_trainer = SFTTrainer(
    model=best_model,
    tokenizer=best_tok,
    train_dataset=train_ds,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    data_collator=DataCollatorForSeq2Seq(tokenizer=best_tok),
    packing=False,
    args=SFTConfig(
        output_dir='outputs/llama3.2-3b-fpb-optuna-best',
        num_train_epochs=best['epochs'],
        per_device_train_batch_size=best['batch_size'],
        gradient_accumulation_steps=best['grad_accum'],
        learning_rate=best['learning_rate'],
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        weight_decay=0.01,
        optim='adamw_8bit',
        logging_steps=20,
        save_strategy='epoch',
        seed=SEED,
        report_to='none',
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
    ),
)

best_trainer = train_on_responses_only(
    best_trainer,
    instruction_part='<|start_header_id|>user<|end_header_id|>\n\n',
    response_part='<|start_header_id|>assistant<|end_header_id|>\n\n',
)

best_train_out = best_trainer.train()
best_model.save_pretrained(BEST_ADAPTER_DIR)
best_tok.save_pretrained(BEST_ADAPTER_DIR)

FastLanguageModel.for_inference(best_model)
best_model.eval()

best_preds = batched_generate_labels(
    best_model,
    best_tok,
    gold_eval_df['sentence'].tolist(),
    batch_size=64,
    max_new_tokens=3,
)

best_metrics = evaluate_predictions(gold_eval_df['label_text'].tolist(), best_preds)
print_metrics('LLaMA3.2-3B qLoRA (Optuna Best) on Gold Commodity', best_metrics)

log_run_to_wandb(
    run_name='llama3.2-3b-qlora_optuna_best_fullfpb_gold',
    config_dict={
        'model_name': LLAMA_MODEL_NAME,
        'adapter_dir': BEST_ADAPTER_DIR,
        'stage': 'finetune_optuna_best',
        'dataset_train': 'financialphrasebank_75agree_full',
        'dataset_eval': 'gold-commodity-sentiment-eval',
        **best,
        'train_loss': float(best_train_out.training_loss),
        'optuna_best_trial': int(study.best_trial.number),
        'optuna_best_macro_f1': float(study.best_value),
    },
    metrics=best_metrics,
)

## 7. Compact Comparison Table

In [ ]:
comparison_df = pd.DataFrame([
    {'model': 'Qwen2.5-3B-Instruct (zero-shot)', **{k: v for k, v in qwen_metrics.items() if isinstance(v, (int, float, np.floating))}},
    {'model': 'LLaMA3.2-3B qLoRA (default)', **{k: v for k, v in llama_metrics.items() if isinstance(v, (int, float, np.floating))}},
    {'model': 'LLaMA3.2-3B qLoRA (optuna best)', **{k: v for k, v in best_metrics.items() if isinstance(v, (int, float, np.floating))}},
])

display_cols = ['model', 'macro_f1', 'micro_f1', 'weighted_f1', 'accuracy', 'mcc', 'f1_negative', 'f1_neutral', 'f1_positive']
display(comparison_df[display_cols].sort_values('macro_f1', ascending=False).reset_index(drop=True))

## 8. Placeholder Save of Best Model


In [ ]:
from pathlib import Path

EXPORT_DIR = Path("experiments/sentiment_agent/saved_models/optuna_best_placeholder")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

if "best_model" in globals() and "best_tok" in globals():
    best_model.save_pretrained(EXPORT_DIR)
    best_tok.save_pretrained(EXPORT_DIR)
    print(f"Saved best model/tokenizer to: {EXPORT_DIR}")
elif "llama_model" in globals() and "llama_tokenizer" in globals():
    llama_model.save_pretrained(EXPORT_DIR)
    llama_tokenizer.save_pretrained(EXPORT_DIR)
    print(f"Saved fallback llama model/tokenizer to: {EXPORT_DIR}")
elif "baseline_model" in globals() and "baseline_tokenizer" in globals():
    baseline_model.save_pretrained(EXPORT_DIR)
    baseline_tokenizer.save_pretrained(EXPORT_DIR)
    print(f"Saved baseline model/tokenizer to: {EXPORT_DIR}")
elif "model" in globals() and "tokenizer" in globals():
    model.save_pretrained(EXPORT_DIR)
    tokenizer.save_pretrained(EXPORT_DIR)
    print(f"Saved generic model/tokenizer to: {EXPORT_DIR}")
else:
    placeholder = EXPORT_DIR / "README_PLACEHOLDER.txt"
    placeholder.write_text(
        "No in-memory model object found. This directory is reserved for the best model export.",
        encoding="utf-8",
    )
    print(f"Created placeholder file: {placeholder}")

## 8. Cleanup (Optional)

In [ ]:
for var_name in ['qwen_model', 'qwen_tokenizer', 'llama_model', 'llama_tokenizer', 'best_model', 'best_tok', 'trainer', 'best_trainer']:
    if var_name in globals():
        del globals()[var_name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Cleanup complete.')